In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
import seaborn as sns

warnings.filterwarnings('ignore')

# 교통인프라 데이터 불러오기

In [ ]:
data=pd.read_csv('/content/인프라지수_자료모음.csv')

In [ ]:
data3=data.copy()

In [ ]:
data3

In [ ]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
import matplotlib.pyplot as plt

plt.rc('font', family='NanumBarunGothic')

In [ ]:
columns = data3.columns.drop('시군구코드')

# 각 변수에 대해 정규분포 그래프 그리기
for col in columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(data3[col].dropna(), kde=True, stat='density', color='skyblue', bins=30)
    plt.title(f"{col}")
    plt.grid(True)
    plt.show()

In [ ]:
data3.skew()

# skew가 3이상인 변수: 정류장밀도, 철도역밀도, 1인당 대중교통 이용량-> 로그 변환 진행

In [ ]:
#로그변환 (왜도가 3이상인 경우에 로그변환진행함)
data3['정류장 밀도'] = np.log1p(data3['정류장 밀도'])
data3['철도역 밀도'] = np.log1p(data3['철도역 밀도'])
data3['1인당 대중교통 이용량'] = np.log1p(data3['1인당 대중교통 이용량'])

In [ ]:
columns = ['정류장 밀도','철도역 밀도','1인당 대중교통 이용량']

# 각 변수에 대해 정규분포 그래프 그리기
for col in columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(data3[col].dropna(), kde=True, stat='density', color='skyblue', bins=30)
    plt.title(f"{col}")
    plt.grid(True)
    plt.show()

# 정규화 진행

In [ ]:
from sklearn.preprocessing import StandardScaler
X = data3.drop(['시군구코드', '발생량'], axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 정규화된 값으로 원래 컬럼 덮어쓰기
data3[X.columns] = X_scaled

# 회귀분석 진행
x: 시군구코드와 발생량을 제외한 변수들
y: 발생량

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression


# X와 y 분리
X = data3.drop(['시군구코드','발생량'],axis=1)
y = data3['발생량']

# 회귀 모델 생성 및 학습
model = LinearRegression()
model.fit(X, y)

# 회귀 계수 및 절편 출력
coefficients = pd.Series(model.coef_, index=X.columns)
intercept = model.intercept_

print("회귀 계수:")
print(coefficients)
print("\n절편 (intercept):", intercept)

# R^2 점수 확인 (모델 설명력)
print("\n설명력 (R^2):", model.score(X, y))

회귀계수를 모두 더한 뒤 각 계수를 정규화(총합으로 나눔)하여 가중치로 활용,
해당 가중치를 변수값에 곱해 만든 "교통인프라수준 지수"를 만듦

In [ ]:
norm_weights = model.coef_ / np.sum(np.abs(model.coef_))  # 절댓값 기준 합 1
data31=data3.copy()
data31['교통인프라지수_norm'] = 0

# 적용
for i in range(len(norm_weights)):
    data31['교통인프라지수_norm'] += norm_weights[i] * X.iloc[:, i]


sc=data31.sort_values(by='교통인프라지수_norm', ascending=False)

In [ ]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# (2) 상수항 추가 (회귀식의 절편 포함)
X_const = add_constant(X)

# (3) VIF 계산
vif_data = pd.DataFrame()
vif_data["변수명"] = X_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]

# 결과 출력
print(vif_data)


다중공산성을 보았을때, 모든변수의 VIF <=10이하임

In [ ]:
result=sc[['시군구코드','교통인프라지수_norm']]
result.to_csv('/content/교통인프라지수_최종(3).csv')